# 02 - Model training and evaluation

Build leakage-safe pre-match features, compare models with rolling-origin validation, calibrate the selected model, and save the fitted artifact.

In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from fifa_predict.data import combine_completed_matches
from fifa_predict.pipeline import audit_sources, fit_model

offline = os.getenv("FIFA_PREDICT_OFFLINE", "").lower() in {"1", "true", "yes"}
audit = audit_sources(refresh=not offline, offline=offline)
matches = combine_completed_matches(audit.historical, audit.world_cup)
bundle, leakage_audit = fit_model(matches, save=True)
bundle.model_version, bundle.trained_through

In [ ]:
for column in ["home_history_cutoff", "away_history_cutoff"]:
    known = leakage_audit[column].notna()
    assert (leakage_audit.loc[known, column].dt.floor("D") < leakage_audit.loc[known, "date"].dt.floor("D")).all()

metrics = pd.DataFrame(bundle.metrics["candidates"]).T
metrics.loc["naive"] = bundle.metrics["naive"]
metrics.sort_values("log_loss")

In [ ]:
plot_data = metrics.reset_index(names="model")
sns.set_theme(style="whitegrid")
ax = sns.barplot(data=plot_data, x="log_loss", y="model", color="#3763a3")
ax.set(title=f"Rolling validation log loss - selected: {bundle.model_name}", xlabel="Multiclass log loss", ylabel="")
plt.show()

bundle.metrics["selected_calibrated"]